# Bill Splitter Agent -- Multi-step Trace Demo

This notebook proves the Bill Splitter is an **agent**, not a scripted chatbot:
given a goal, it decides *which* tools to call, *in what order*, and *how many times*,
without that sequence being hardcoded anywhere in this notebook.

It imports the real project code (`agent.py`, `tools.py`, `memory.py`) from this
same folder -- this is not a reimplementation, it's the graded code running live.

## Setup

Requires a `.env` file (see `.env.example`) with `GROQ_API_KEY=...` in this same folder.

In [1]:
import sys
sys.path.insert(0, ".")

from dotenv import load_dotenv
load_dotenv()

from memory import BillMemory
from tools import BillTools
from agent import BillSplitterAgent

print("imports ok")

imports ok


## Part 1: Sanity-check the tools directly (no API key needed)

Before ever trusting the model to call these correctly, we call `add_share` and
`settle_up` ourselves. This is exactly the step that was skipped originally and
caused the bug described in the README -- `settle_up()` crashed because it called
a method that didn't exist. Testing tools in isolation first is how that class of
bug gets caught before it reaches the agent.

In [2]:
memory = BillMemory()
tools = BillTools(memory)

print("add_share(Rahul, 900):")
print(tools.add_share("Rahul", 900))

print()
print("add_share(Aman, 600):")
print(tools.add_share("Aman", 600))

print()
print("settle_up():")
print(tools.settle_up())

add_share(Rahul, 900):
{'success': True, 'message': 'Rahul paid ₹900.00', 'expenses': [{'person': 'Rahul', 'amount': 900.0}]}

add_share(Aman, 600):
{'success': True, 'message': 'Aman paid ₹600.00', 'expenses': [{'person': 'Rahul', 'amount': 900.0}, {'person': 'Aman', 'amount': 600.0}]}

settle_up():
{'success': True, 'total': 1500.0, 'fair_share': 750.0, 'people': {'Rahul': 900.0, 'Aman': 600.0}, 'balances': {'Rahul': 150.0, 'Aman': -150.0}, 'settlements': [{'from': 'Aman', 'to': 'Rahul', 'amount': 150.0}]}


## Part 2: Run the actual agent on 2-3 example goals

Each `agent.chat(goal, verbose=True)` call prints the step-by-step trace:
every tool the model chose to call, the arguments it picked, and the tool's
result -- before the model's final natural-language answer.

**Run the cells below yourself** with a valid `GROQ_API_KEY` in `.env` to produce
the live trace output for submission -- it isn't pre-baked here since it depends
on a real model call.

### Goal 1: Multiple payments reported in one message, then a settlement request

The agent must decide, unprompted, to call `add_share` twice and `settle_up` once,
in that order -- the order is not specified by us, only implied by the goal text.

In [3]:
memory = BillMemory()
tools = BillTools(memory)
agent = BillSplitterAgent(tools)

goal_1 = (
    "Rahul paid Rs 900 for our shared dinner. "
    "Aman paid Rs 600 for the same dinner. "
    "Now settle the bill and tell us who owes whom."
)

answer = agent.chat(goal_1, verbose=True)
print()
print("Final answer:", answer)

[step 1] add_share({'amount': 900, 'person': 'Rahul'}) -> {'success': True, 'message': 'Rahul paid ₹900.00', 'expenses': [{'person': 'Rahul', 'amount': 900.0}]}
[step 2] add_share({'amount': 600, 'person': 'Aman'}) -> {'success': True, 'message': 'Aman paid ₹600.00', 'expenses': [{'person': 'Rahul', 'amount': 900.0}, {'person': 'Aman', 'amount': 600.0}]}
[step 3] settle_up({}) -> {'success': True, 'total': 1500.0, 'fair_share': 750.0, 'people': {'Rahul': 900.0, 'Aman': 600.0}, 'balances': {'Rahul': 150.0, 'Aman': -150.0}, 'settlements': [{'from': 'Aman', 'to': 'Rahul', 'amount': 150.0}]}
[step 4] final answer

Final answer: Here’s what happened:

| Person | Amount Paid | Fair Share (₹) | Net Balance |
|--------|-------------|----------------|-------------|
| Rahul  | ₹900.00     | ₹750.00        | +₹150.00    |
| Aman   | ₹600.00     | ₹750.00        | –₹150.00    |

**Settlement**

- Aman owes Rahul **₹150.00**.  
- After this payment, everyone will have contributed equally.

Let me k

### Goal 2: Memory across separate turns

Two payments are reported in two *separate* messages, then a third message asks
for settlement with no new payment info. This proves `BillMemory` -- not the
prompt text -- is what the settlement is computed from.

In [4]:
memory = BillMemory()
tools = BillTools(memory)
agent = BillSplitterAgent(tools)

print("Turn 1:")
print(agent.chat("Rahul paid Rs 300 for snacks.", verbose=True))

print()
print("Turn 2:")
print(agent.chat("Aman paid Rs 200 for snacks.", verbose=True))

print()
print("Turn 3:")
print(agent.chat("Settle everything.", verbose=True))

Turn 1:
[step 1] add_share({'amount': 300, 'person': 'Rahul'}) -> {'success': True, 'message': 'Rahul paid ₹300.00', 'expenses': [{'person': 'Rahul', 'amount': 300.0}]}
[step 2] final answer
Got it—Rahul paid ₹300 for the snacks. Let me know the details of everyone involved and any other payments so I can help calculate the splits and who owes whom.

Turn 2:
[step 1] add_share({'amount': 200, 'person': 'Aman'}) -> {'success': True, 'message': 'Aman paid ₹200.00', 'expenses': [{'person': 'Rahul', 'amount': 300.0}, {'person': 'Aman', 'amount': 200.0}]}
[step 2] final answer
Thanks! Rahul paid ₹300 and Aman paid ₹200 for the snacks. If there are other people who shared the cost or any additional expenses, let me know. Once I have all the details, I can calculate the fair share and show who owes whom.

Turn 3:
[step 1] settle_up({}) -> {'success': True, 'total': 500.0, 'fair_share': 250.0, 'people': {'Rahul': 300.0, 'Aman': 200.0}, 'balances': {'Rahul': 50.0, 'Aman': -50.0}, 'settlements':

### Goal 3: Settlement-only request, no new payment

The agent should call `settle_up` and should **not** call `add_share`, since no
payment was mentioned -- showing it isn't just calling every tool every time.

In [5]:
print(agent.chat("Who owes money right now?", verbose=True))

[step 1] final answer
Right now, **Aman owes ₹50**. Rahul has a net credit of ₹50, so once that payment is made, the balances will be even.
